# Data Preprocessing

## Define the independent variables as X and the dependent as Y

In [ ]:
X = train.drop('price', axis = 1)
y = train['price']

### Types correction

In the EDA phase we noticed some inconsistencies in the variables types. We will further decide what to do with this errors and correct them.

In [ ]:
# year, previousOwners as float
(X['year'] % 1 != 0).value_counts() # the true values are the incorrect ones
(X['previousOwners'] % 1 != 0).value_counts() # the true values are the incorrect ones

#percentage of incorrect values
num_incorrect_year = (X['year'] % 1 != 0).sum() / X.shape[0] * 100
print(f'Percentage of incorrect values in year column: {num_incorrect_year:.2f}%')
num_incorrect_previousOwners = (X['previousOwners'] % 1 != 0).sum() / X.shape[0] * 100
print(f'Percentage of incorrect values in previousOwners column: {num_incorrect_previousOwners:.2f}%')



We decide to round all values of the features even if the percentage of incorrect close to 3%.

In [ ]:
X = X.copy()
#'Int64' is used to keep missing values and still use the round and asType

X['year'] = X['year'].round().astype('Int64')

X['previousOwners'] = X['previousOwners'].round().astype('Int64')

X['hasDamage'] = X['hasDamage'].astype('Int64') #this variable only has 0s and 1s but it was in a float type


### Values Treatment

We already notice that we have some negative values on the feature mileage, tax, mpg, engineSize and previousOwner that make no sense.

What should we do with this negative values? First of all we'll analyse their percentage:

In [ ]:
mileage_train_negatives = X['mileage']<0
tax_train_negatives = X['tax']<0
mpg_train_negatives = X['mpg']<0
engineSize_train_negatives = X['engineSize']<0
previousOwners_train_negatives = X['previousOwners']<0


#here we used chatGPT to help us to construct the following DataFrame
negatives_summary = pd.DataFrame({
    'mileage_negatives': mileage_train_negatives.value_counts(),
    'tax_negatives': tax_train_negatives.value_counts(),
    'mpg_negatives': mpg_train_negatives.value_counts(),
    'engineSize_negatives': engineSize_train_negatives.value_counts(),
    'previousOwners_negatives': previousOwners_train_negatives.value_counts()
})

negatives_summary

In [ ]:
#Check the percentage of negative mileage values
data_len = len(train['mileage'])

#Here we used chatGPT to help us constructing the final negatives_table
negatives_percent = {
    'mileage_negatives (%)': mileage_train_negatives.sum() / data_len * 100,
    'tax_negatives (%)': tax_train_negatives.sum() / data_len * 100,
    'mpg_negatives (%)': mpg_train_negatives.sum() / data_len * 100,
    'engineSize_negatives (%)': engineSize_train_negatives.sum() / data_len * 100,
    'previousOwners_negatives (%)': previousOwners_train_negatives.sum() / data_len * 100
}

# Converter para DataFrame (tabela)
negatives_table = pd.DataFrame(negatives_percent, index=['Percentage of Negatives']).round(3)

negatives_table

From the negatives_table we obtain very low percentages for each variable, lower than 1%. So it's possible to conclude that this values are a small part of our data and that if we drop them we don't loose veracity.

We can also see that the total percentage of negative values is lower then 2%.

The first two approaches that came in our minds where drop this negative values or turn them into their absolute value.

We decide to take the 2nd approach.

In [ ]:
X['mileage'] = X['mileage'].abs()
X['tax'] = X['tax'].abs()
X['mpg'] = X['mpg'].abs()
X['engineSize'] = X['engineSize'].abs()
X['previousOwners'] = X['previousOwners'].abs() 


Th rest of the data still have some incosistencies in features like PaintQuality% that is a percentage and has values greater than 100%. 

To deal with this we start to check the percentage of this errors.

In [ ]:
(X['paintQuality%']>100).value_counts() # the true values are the incorrect ones

percentage = (X['paintQuality%']>100).sum() / X.shape[0] * 100
print(f'Percentage of incorrect values in paintQuality% column: {percentage:.2f}%')


In the HasDamage column, we have 0's and NaN and we decided to fill the NaN's with 1's to creat a boolean variable which will be easier for next analysis.

In [ ]:
train['hasDamage']=train['hasDamage'].fillna(1)

With this result we can treat the incorrect values as NaN or drop them.

TO DO:
- corrigir types
- remover outliers graves, que são erros e visualizar boxplots
- Feature engineering: Criar features que não envolvem cálculos c/ média, mediana, …
Fazer tudo isto para o training and test set

### Split the dataset into train and validation

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.3, 
                                                  random_state = 0, 
                                                  #stratify = y- não pus esta parte como no notebook pq estava a dar erro e acho que é pq a nossa variavel y aqui não é um boolean mas sim um float
                                                  shuffle = True)

TO DO:

- Feature engineering: Criar features que envolvem cálculos c/ média, mediana, … (fazer para os 3 sets que temos)
- Preencher missing values nos 3 datasets, acho que dá para usar uma função ‘transform’ (justificar para depois por no report)  
!!!!!ATENÇÃO: os valores de média, moda,… a usar neste últimos dois passos são todos retirados apenas do training set e não do validation set

### Fill missing values

As we saw there are missing values in a couple of variables so we will fill the categories missing values with 'Unknow' and the numericals with the average.

But before do that we'll split our columns in metric and non_metric features.

In [ ]:
for column in ['Brand', 'model', 'transmission', 'fuelType']:
    X_train[column] = X_train[column].fillna('Unknown')
    X_val[column] = X_val[column].fillna('Unknown')

In [ ]:
for column in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[column]):
        
        #store mean of training data in a variable - in a real application, you may need to store these values for future usages on e.g. test data 
        mean_to_fill = X_train[column].mean()
        
        #fill on X_train
        X_train[column].fillna(mean_to_fill, inplace=True)
        #Fill on X_val
        X_val[column].fillna(mean_to_fill, inplace=True)

## Test

In [ ]:
test.shape
#Here we can see that the test shape is equal to the sample shape.

In [ ]:
test.head(15)

In [ ]:
test.tail(15)

From the visualization of the head and tail of the data base we can already understand that some errors exist:

    - Missing values
    - Values in the columns Year, hasDamage, previousOwners that should be integers as floats (2020.0)
    - Floats on mpg, previousOwners column with diferent sizes
    - A category unknown in transmission column
    - It looks like the column hasDamage only contains 0's and blanks/None values, are the blanks supose to be 1's?
    - PreviousOwner and tax: negative values are impossible
We will further analyse this using describe and info.

It's also possible to see that some strings have the same information written in different forms (Petrol as etrol, Automatic and AUTOMATIC).
To solve this problem we will uniformize all the values in data preparation

In [ ]:
test.info()

From info we can see that:

    - year, as a float...
    - previousOwners, hasDamage also as floats but they should be integers and booleans respectively
    - Missing values in all features

What will we do?

    Analyse with describe to have a different view

In [ ]:
test.describe()

From the numeric describe we can see that we have some weird values:

    1. negative mileage, tax, mpg, engineSize, previousOwners in the minimum value
    2. hasDamage is a boolean but we can see that instead of 0 and 1 we only have 0 and Nones*
    3. previousOwner has a float? Should we round it?

*check in the hasDamage column

What will we do:

    1. Count the number of negative values and decide if we should drop or change them.
    2. Replace the nones by 1's. (data-preparation)
    3. Count the number of float values and decide to drop or round them.

In [ ]:
test.describe(include='object')

From the categorical describe we tell that:

    - This columns also have missing values
    - transmission has 38 unique values and fuelType has 29...


In [ ]:
brand_same_size = train['Brand'].str.lower()
brand_same_size.value_counts().to_frame()

In [ ]:
mapping_brands = {
    'ord' : 'ford',
    'for' : 'ford',
    'ercedes' : 'mercedes',
    'mercede' : 'mercedes',
    'w' : 'vw',
    'v' : 'vw',
    'ope' : 'opel',
    'pel' : 'opel',
    'mw' : 'bmw',
    'aud' : 'audi',
    'udi' : 'audi',
    'bm' : 'bmw',
    'oyota' : 'toyota',
    'koda' : 'skoda',
    'skod' : 'skoda',
    'toyot' : 'toyota',
    'yundai' : 'hyundai',
    'hyunda' : 'hyundai',
    'ercede' : 'mercedes',
    'or' : 'ford',
    'pe' : 'opel',
    'yunda' : 'hyundai',
    'ud' : 'audi',
    'kod' : 'skoda'
}
#train['Brand'] = train['Brand].replace(mapping_brands)

podemos usar esta maneira ou uma biblioteca chamada fuzzy matching mas não sei se vai ser permitido

previousOwner column - round to transform float into int

In [ ]:
train['previousOwners'] = train['previousOwners'].round(0) #do the same for test set?

## Prepare Feature Engineering

### Normalize the categorical variables

- fazer tanto para o train como para o validation mas usando só os valores do train